In [23]:
from dotenv import load_dotenv
load_dotenv()
import os
os.environ["GROQ_API"] = "your_actual_key_here"

In [37]:
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

In [39]:
loader = PyPDFLoader('./Data/medical_report.pdf')
docs = loader.load()
len(docs)

9

In [40]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)
splitted_data = splitter.split_documents(docs)
len(splitted_data)

48

In [41]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

In [42]:
vector_store = Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings
)

In [43]:
query = "Machine Learnings"
data = vector_store.similarity_search(query=query)
len(data)
data[0].page_content

'Customer Segmentation\nE-commerce Product Clustering\n🎤  Mock Interview: Unsupervised Learning \ue09d Features\n✅  Module 8: Time Series + Intro to Deep Learning (3 \nweeks)\nDuration: Month 8\nTopics:\nTime Series components, lag, rolling stats\nForecasting: ARIMA, Prophet\nIntro to Neural Networks\nPerceptron, basic Keras model\nTools:\nStatsmodels, Prophet, Keras, TensorFlow\nMini Project:\nStock Price or Weather Forecasting\n🎤  Mock Interview: Forecasting \ue09d Neural Net Basics'

In [45]:
context = ""
for doc in data:
    context += doc.page_content + '\n'

In [46]:
llm = ChatGroq(model="openai/gpt-oss-120b")

In [34]:
res = llm.invoke(f"""can you provide me the answer based on the provided context for my questions,
    context:{context} and question:{query}
""")

In [47]:
def getContext(query:str):
    data = vector_store.similarity_search(query=query)
    context = ""
    for doc in data:
        context += doc.page_content + '\n'

    return {
        "context":context,
        "question":query
    }

In [48]:
prompt = PromptTemplate.from_template("""
    You are a helpful assistant and provide answerd based on the context for user question. and 
    if you don't know the answer, then you can say that 'I dont know.'
    Context:{context}
    Question:{question}
""")

In [49]:
rag_chain = getContext | prompt | llm

In [63]:
res = rag_chain.invoke("what is the name of patient?")

In [64]:
print(res.content)

The patient’s name is **Ms. Nikita Chudhary**.
